In [1]:
import sys
import introns
import importlib
import warnings

import numpy as np
import pandas as pd
import xgboost as xgb
import matplotlib.pyplot as plt

from Bio import SeqIO
from Bio.SeqUtils import GC
from termcolor import colored
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.multioutput import MultiOutputClassifier
from sklearn.metrics import accuracy_score, classification_report, precision_recall_fscore_support, multilabel_confusion_matrix

importlib.reload(introns)

<module 'introns' from '/home/semik/projekty/ml_attempts/noncanonical_introns_ml_solutions_2/introns.py'>

## Trenowanie
4 klasy: konw, niekonw, inne pozycje, losowe sekwencje

In [2]:
### sprawdzanie ze introny nie sa swoimi wlasnymi wariantami

seqs = [i for i in SeqIO.parse("introns_for_4-class_classifier_training.fasta", 'fasta')]
for id1 in range(len(seqs)-1):
    s1 = seqs[id1]
    for id2 in range(id1+1, len(seqs)):
        s2 = seqs[id2]
        if s1.seq==s2.seq:
            print(s1.description)
            print(s2.description)
            print('\n')

In [3]:
introns_seqs, introns_classes, introns_characteristics = [], [], []
f = open("introns_for_4-class_classifier_training.fasta", 'r')
for l in f.readlines():
    if l.startswith(">"):
        t = l.split()[-1]
        intron_type = 0 if t=="intron_K" else (1 if t=="intron_NK" else (2 if t=="intron_other" else 3))
    else:
        prev_exon_seq, intron_seq, next_exon_seq = l[:5], l[5:-5], l[-5:]
        its_characteristics = introns.compute_intron_characteristics(l)
        introns_seqs.append((prev_exon_seq, intron_seq, next_exon_seq))
        introns_classes.append(intron_type)
        introns_characteristics.append(its_characteristics)
print(len(introns_seqs), len(introns_classes), len(introns_characteristics))

X_train, X_test, y_train, y_test = train_test_split(introns_characteristics, introns_classes, test_size=0.3, random_state=420)
introns_classes = np.array(introns_classes).reshape(-1,1)

2683 2683 2683


In [4]:
clf_4head = xgb.XGBClassifier(use_label_encoder=False)
clf_4head.fit(X_train, y_train);

pred_4head = clf_4head.predict(X_test)

In [5]:
print("4-class classifier accuracy score:", accuracy_score(y_test,pred_4head))

precision, recall, fscore, _ = precision_recall_fscore_support(y_test,pred_4head)
print("Precision:\t", precision)
print("recall:\t\t", recall, fscore)
print("f-score:\t", fscore)
print(multilabel_confusion_matrix(y_test,pred_4head))

4-class classifier accuracy score: 0.6422360248447205
Precision:	 [0.44444444 0.46341463 0.65425532 0.66666667]
recall:		 [0.03846154 0.13475177 0.97233202 0.03703704] [0.07079646 0.20879121 0.78219396 0.07017544]
f-score:	 [0.07079646 0.20879121 0.78219396 0.07017544]
[[[696   5]
  [100   4]]

 [[642  22]
  [122  19]]

 [[ 39 260]
  [ 14 492]]

 [[750   1]
  [ 52   2]]]


In [7]:
from sklearn import metrics
xgbcl = xgb.XGBClassifier(use_label_encoder=False)
#parameters={'booster':['gbtree', 'gblinear', 'dart'], 'sampling_method':['uniform', 'gradient_based'],
#            'importance_type':['gain', 'weight', 'cover', 'total_gain', 'total_cover'], 'enable_categorical':[True, False],
#            'eval_metric':['metrics.accuracy_score', 'metrics.average_precision_score', 'metrics.f1_score', 'metrics.log_loss', 'metrics.precision_score', 'metrics.recall_score', 'metrics.jaccard_score', 'metrics.balanced_accuracy_score', 'metrics.brier_score_loss', 'metrics.top_k_accuracy_score', 'metrics.roc_auc_score'],
#            'tree_method':['updater','tree_method', 'exact', 'approx', 'hist', 'gpu_hist', 'grow_local_histmaker', 'refresh', 'prune', 'sync','updater', 'tree_method']}
parameters={'booster':['gbtree', 'gblinear', 'dart'],
            'sampling_method':['uniform', 'gradient_based'],
            'importance_type':['gain', 'weight', 'cover', 'total_gain', 'total_cover'],
            'enable_categorical':[True, False],
            'tree_method':['updater','tree_method', 'exact', 'approx', 'hist', 'grow_local_histmaker', 'refresh', 'prune', 'updater', 'tree_method']}
clf_4head_gs = GridSearchCV(xgbcl, parameters)
clf_4head_gs.fit(X_train, y_train)
pred_4head_gs = clf_4head_gs.predict(X_test)

[14:28:00] WARNING: ../src/learner.cc:627: 
Parameters: { "sampling_method", "tree_method" } might not be used.

  This could be a false alarm, with some parameters getting used by language bindings but
  then being mistakenly passed down to XGBoost core, or some parameter actually being used
  but getting flagged wrongly here. Please open an issue if you find any such cases.


[14:28:01] WARNING: ../src/learner.cc:627: 
Parameters: { "sampling_method", "tree_method" } might not be used.

  This could be a false alarm, with some parameters getting used by language bindings but
  then being mistakenly passed down to XGBoost core, or some parameter actually being used
  but getting flagged wrongly here. Please open an issue if you find any such cases.


[14:28:01] WARNING: ../src/learner.cc:627: 
Parameters: { "sampling_method", "tree_method" } might not be used.

  This could be a false alarm, with some parameters getting used by language bindings but
  then being mistakenly passed down

/home/semik/miniconda3/lib/python3.8/site-packages/sklearn/model_selection/_validation.py:372: FitFailedWarning: 
2150 fits failed out of a total of 3300.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
1350 fits failed with the following error:
Traceback (most recent call last):
  File "/home/semik/miniconda3/lib/python3.8/site-packages/sklearn/model_selection/_validation.py", line 681, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/home/semik/miniconda3/lib/python3.8/site-packages/xgboost/core.py", line 532, in inner_f
    return f(**kwargs)
  File "/home/semik/miniconda3/lib/python3.8/site-packages/xgboost/sklearn.py", line 1379, in fit
    model, metric, params, early_stopping_rounds, callbacks = self._c

[14:55:18] WARNING: ../src/learner.cc:627: 
Parameters: { "sampling_method", "tree_method" } might not be used.

  This could be a false alarm, with some parameters getting used by language bindings but
  then being mistakenly passed down to XGBoost core, or some parameter actually being used
  but getting flagged wrongly here. Please open an issue if you find any such cases.




In [8]:
print("Optimized 4-class classifier accuracy score:", accuracy_score(y_test,pred_4head_gs))

precision, recall, fscore, _ = precision_recall_fscore_support(y_test,pred_4head_gs)
print("Precision:\t", precision)
print("recall:\t\t", recall)
print("f-score:\t", fscore)
print(multilabel_confusion_matrix(y_test,pred_4head_gs))

Optimized 4-class classifier accuracy score: 0.6335403726708074
Precision:	 [0.         0.         0.63204005 0.83333333]
recall:		 [0.         0.         0.99802372 0.09259259]
f-score:	 [0.         0.         0.77394636 0.16666667]
[[[701   0]
  [104   0]]

 [[664   0]
  [141   0]]

 [[  5 294]
  [  1 505]]

 [[750   1]
  [ 49   5]]]


/home/semik/miniconda3/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1308: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


In [11]:
pred_4head_gs

array([2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
       2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
       2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
       2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
       2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
       2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
       2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 3, 2,
       2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
       2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
       2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
       2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 3, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
       2, 2, 2, 3, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
       2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
       2, 2, 2, 2, 2, 3, 2, 2, 2, 2, 2, 2, 2, 2, 2,

## Trenowanie
3 klasy: konw, niekonw, inne pozycje

In [12]:
### sprawdzanie ze introny nie sa swoimi wlasnymi wariantami

seqs = [i for i in SeqIO.parse("introns_for_3-class_classifier_training.fasta", 'fasta')]
for id1 in range(len(seqs)-1):
    s1 = seqs[id1]
    for id2 in range(id1+1, len(seqs)):
        s2 = seqs[id2]
        if s1.seq==s2.seq:
            print(s1.description)
            print(s2.description)
            print('\n')

In [13]:
introns_seqs, introns_classes, introns_characteristics = [], [], []
f = open("introns_for_3-class_classifier_training.fasta", 'r')
for l in f.readlines():
    if l.startswith(">"):
        t = l.split()[-1]
        intron_type = 0 if t=="intron_K" else (1 if t=="intron_NK" else 2)
    else:
        prev_exon_seq, intron_seq, next_exon_seq = l[:5], l[5:-5], l[-5:]
        its_characteristics = introns.compute_intron_characteristics(l)
        introns_seqs.append((prev_exon_seq, intron_seq, next_exon_seq))
        introns_classes.append(intron_type)
        introns_characteristics.append(its_characteristics)
print(len(introns_seqs), len(introns_classes), len(introns_characteristics))

introns_classes = np.array(introns_classes).reshape(-1,1)

X_train, X_test, y_train, y_test = train_test_split(introns_characteristics, introns_classes, test_size=0.3, random_state=420)

2483 2483 2483


In [14]:
clf_3head = MultiOutputClassifier(xgb.XGBClassifier(use_label_encoder=False))
clf_3head.fit(X_train, y_train);

pred_3head = clf_3head.predict(X_test)

In [15]:
print("3-class classifier accuracy score:", accuracy_score(y_test,pred_3head))

precision, recall, fscore, _ = precision_recall_fscore_support(y_test,pred_3head)
print("Precision:\t", precision)
print("recall:\t\t", recall)
print("f-score:\t", fscore)
print(multilabel_confusion_matrix(y_test,pred_3head))

3-class classifier accuracy score: 0.6738255033557047
Precision:	 [0.5        0.59259259 0.67937853]
recall:		 [0.04716981 0.10666667 0.98364008]
f-score:	 [0.0862069  0.18079096 0.80367586]
[[[634   5]
  [101   5]]

 [[584  11]
  [134  16]]

 [[ 29 227]
  [  8 481]]]


In [21]:
parameters={'booster':['gbtree', 'gblinear', 'dart'],
            'sampling_method':['uniform', 'gradient_based'],
            'importance_type':['gain', 'weight', 'cover', 'total_gain', 'total_cover'],
            'enable_categorical':[True, False],
            'tree_method':['updater','tree_method', 'exact', 'approx', 'hist', 'grow_local_histmaker', 'refresh', 'prune', 'updater', 'tree_method']}

xgbcl = xgb.XGBClassifier(use_label_encoder=False)
clf_3head_gs = GridSearchCV(xgbcl, parameters)
clf_3head_gs.fit(X_train, y_train)
pred_3head_gs = clf_3head_gs.predict(X_test)

[18:51:11] WARNING: ../src/learner.cc:627: 
Parameters: { "sampling_method", "tree_method" } might not be used.

  This could be a false alarm, with some parameters getting used by language bindings but
  then being mistakenly passed down to XGBoost core, or some parameter actually being used
  but getting flagged wrongly here. Please open an issue if you find any such cases.


[18:51:12] WARNING: ../src/learner.cc:627: 
Parameters: { "sampling_method", "tree_method" } might not be used.

  This could be a false alarm, with some parameters getting used by language bindings but
  then being mistakenly passed down to XGBoost core, or some parameter actually being used
  but getting flagged wrongly here. Please open an issue if you find any such cases.


[18:51:13] WARNING: ../src/learner.cc:627: 
Parameters: { "sampling_method", "tree_method" } might not be used.

  This could be a false alarm, with some parameters getting used by language bindings but
  then being mistakenly passed down

/home/semik/miniconda3/lib/python3.8/site-packages/sklearn/model_selection/_validation.py:372: FitFailedWarning: 
1900 fits failed out of a total of 3000.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
1200 fits failed with the following error:
Traceback (most recent call last):
  File "/home/semik/miniconda3/lib/python3.8/site-packages/sklearn/model_selection/_validation.py", line 681, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/home/semik/miniconda3/lib/python3.8/site-packages/xgboost/core.py", line 532, in inner_f
    return f(**kwargs)
  File "/home/semik/miniconda3/lib/python3.8/site-packages/xgboost/sklearn.py", line 1379, in fit
    model, metric, params, early_stopping_rounds, callbacks = self._c

[19:13:14] WARNING: ../src/learner.cc:627: 
Parameters: { "sampling_method", "tree_method" } might not be used.

  This could be a false alarm, with some parameters getting used by language bindings but
  then being mistakenly passed down to XGBoost core, or some parameter actually being used
  but getting flagged wrongly here. Please open an issue if you find any such cases.




In [22]:
print("Optimized 3-class classifier accuracy score:", accuracy_score(y_test,pred_3head_gs))

precision, recall, fscore, _ = precision_recall_fscore_support(y_test,pred_3head_gs)
print("Precision:\t", precision)
print("recall:\t\t", recall)
print("f-score:\t", fscore)
print(multilabel_confusion_matrix(y_test,pred_3head_gs))

Optimized 3-class classifier accuracy score: 0.6563758389261745
Precision:	 [0.         0.         0.65637584]
recall:		 [0. 0. 1.]
f-score:	 [0.         0.         0.79254457]
[[[639   0]
  [106   0]]

 [[595   0]
  [150   0]]

 [[  0 256]
  [  0 489]]]


/home/semik/miniconda3/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1308: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


## Trenowanie
4 klasy: konw, niekonw, intermediate, losowe sekwencje

## Trenowanie - multiklasy